# Task 1 — GeoNet search

What the pipeline does first: ask GeoNet for recent events, apply the
processing floor from `auto_tdmt.cfg` §1, and fetch one event's origin.
Every function here is the one the pipeline itself runs
(`geonet.py`, `trigger.py`, `watch.py`).

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "docs" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)
# work in a scratch archive so the real events/ are untouched
os.environ.setdefault("AUTO_TDMT_EVENTS", str(Path.home() / "work" / "proj_tdmt_NZ" / "notebook_runs"))
import config
from config import P            # every tunable, from auto_tdmt.cfg
EVENT = "2026p669681"
print("parameters from", P.source)

In [ ]:
print(json.dumps(P.as_dict()["geonet"], indent=2))

## 1.1 Recent events above the floor

In [ ]:
import trigger
from geonet import recent_quakes
events = recent_quakes(mmi=3)                 # one request to the quake API
print(f"{len(events)} recent events from the quake API")
for ev in events[:15]:
    ok, why = trigger.passes_processing_floor(ev)
    print(f"{ev.public_id} M{ev.prelim_mag:.1f} {ev.depth_km:5.1f} km "
          f"{ev.locality[:32]:32s} -> {'PROCESS' if ok else why}")

## 1.2 One event's origin (the input to task 2)

In [ ]:
from geonet import get_event
ev = get_event(EVENT)
print(ev)
print(json.dumps(ev.to_dict(), indent=2))

## What to check
- Is the floor doing what you expect (`geonet.minPrelimMag`, `maxDepthKm`)?
- Depths reported as 5 / 12 / 33 km are fixed default values rather than
  fitted depths — those events are always attempted and the depth search
  decides (task 3).